# Category-Disjoint Cross-Validation: DeBERTa versus Linear SVM

This is the main Assignment 2 reality-check experiment. It evaluates a
paper-inspired `microsoft/deberta-v3-base` detector and the Assignment 1
TF-IDF unigram Linear SVM using the same five category-disjoint grouped folds.

The protocol keeps each benign email and its injected counterpart in the same
context group, while ensuring that validation attack categories are absent from
the corresponding training fold. The notebook saves fold metrics,
out-of-fold predictions, and comparison tables under `results/`.


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = Path(
        "/content/drive/MyDrive/CSBP711_Assignment_2_Paper_Critique"
    )
else:
    PROJECT_DIR = Path.cwd().resolve().parent
    if not (PROJECT_DIR / "data").exists():
        PROJECT_DIR = Path.cwd().resolve()

DATA_DIR = PROJECT_DIR / "data"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

print("Project directory:", PROJECT_DIR)
print("Training file exists:", TRAIN_PATH.exists())
print("Testing file exists:", TEST_PATH.exists())

import pandas as pd

CV_RESULTS_DIR = PROJECT_DIR / "results" / "grouped_cv_deberta"
CV_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(TRAIN_PATH)

TEXT_COLUMN = "text"
LABEL_COLUMN = "label"
GROUP_COLUMN = "context_id"
SEED = 42

print("Training shape:", train_df.shape)
print("Label distribution:")
print(train_df["label_name"].value_counts())
print("Number of unique contexts:", train_df[GROUP_COLUMN].nunique())
display(train_df.head(4))


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5

context_summary = train_df.groupby(GROUP_COLUMN).agg(
    rows=(LABEL_COLUMN, "size"),
    unique_labels=(LABEL_COLUMN, "nunique")
)

invalid_contexts = context_summary[
    (context_summary["rows"] != 2) |
    (context_summary["unique_labels"] != 2)
]

print("Contexts with exactly one benign and one attack sample:",
      (context_summary["rows"] == 2).sum())

print("Invalid contexts:", len(invalid_contexts))

assert len(invalid_contexts) == 0, (
    "Each context must contain exactly two samples with both labels."
)

cv_splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

fold_assignments = pd.Series(
    index=train_df.index,
    dtype="int64"
)

fold_summary_rows = []

for fold_number, (fit_indices, validation_indices) in enumerate(
    cv_splitter.split(
        X=train_df[TEXT_COLUMN],
        y=train_df[LABEL_COLUMN],
        groups=train_df[GROUP_COLUMN]
    ),
    start=1
):
    fold_assignments.loc[validation_indices] = fold_number

    fit_fold = train_df.iloc[fit_indices]
    validation_fold = train_df.iloc[validation_indices]

    fit_contexts = set(fit_fold[GROUP_COLUMN])
    validation_contexts = set(validation_fold[GROUP_COLUMN])

    context_overlap = fit_contexts.intersection(validation_contexts)

    assert len(context_overlap) == 0, (
        f"Context leakage found in fold {fold_number}."
    )

    fold_summary_rows.append({
        "fold": fold_number,
        "fit_rows": len(fit_fold),
        "validation_rows": len(validation_fold),
        "fit_contexts": len(fit_contexts),
        "validation_contexts": len(validation_contexts),
        "fit_benign": int((fit_fold[LABEL_COLUMN] == 0).sum()),
        "fit_attack": int((fit_fold[LABEL_COLUMN] == 1).sum()),
        "validation_benign": int(
            (validation_fold[LABEL_COLUMN] == 0).sum()
        ),
        "validation_attack": int(
            (validation_fold[LABEL_COLUMN] == 1).sum()
        ),
        "context_overlap": len(context_overlap)
    })

fold_summary = pd.DataFrame(fold_summary_rows)

train_df_cv = train_df.copy()
train_df_cv["cv_fold"] = fold_assignments

FOLD_ASSIGNMENTS_PATH = CV_RESULTS_DIR / "grouped_cv_fold_assignments.csv"
FOLD_SUMMARY_PATH = CV_RESULTS_DIR / "grouped_cv_fold_summary.csv"

train_df_cv.to_csv(FOLD_ASSIGNMENTS_PATH, index=False)
fold_summary.to_csv(FOLD_SUMMARY_PATH, index=False)

print("\nGrouped cross-validation folds created successfully.")
display(fold_summary)

print("\nSamples assigned to each validation fold:")
print(train_df_cv["cv_fold"].value_counts().sort_index())

print(f"\nSaved fold assignments: {FOLD_ASSIGNMENTS_PATH}")
print(f"Saved fold summary: {FOLD_SUMMARY_PATH}")

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available.")

In [ ]:
!pip -q install -U "transformers" "datasets" "accelerate" "evaluate" "sentencepiece==0.2.1" "protobuf==7.36.1"

In [ ]:
from transformers import AutoTokenizer, AutoConfig

MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

model_config = AutoConfig.from_pretrained(MODEL_NAME)

MAX_LENGTH = model_config.max_position_embeddings

print("Tokenizer loaded successfully.")
print("Model maximum position embeddings:", MAX_LENGTH)
print("CLS token:", tokenizer.cls_token, tokenizer.cls_token_id)
print("SEP token:", tokenizer.sep_token, tokenizer.sep_token_id)
print("PAD token:", tokenizer.pad_token, tokenizer.pad_token_id)

In [ ]:
token_lengths = train_df[TEXT_COLUMN].apply(
    lambda text: len(
        tokenizer.encode(
            str(text),
            add_special_tokens=False
        )
    )
)

print("Token length summary:")
display(token_lengths.describe())

long_samples = train_df.loc[
    token_lengths > (MAX_LENGTH - 2),
    [
        "sample_id",
        "context_id",
        "label",
        "label_name",
        "attack_category"
    ]
].copy()

long_samples["token_length"] = token_lengths[
    token_lengths > (MAX_LENGTH - 2)
].values

print(f"\nSamples longer than {MAX_LENGTH - 2} content tokens:")
display(long_samples)

In [ ]:
from torch.utils.data import Dataset

BATCH_SIZE = 4


def head_tail_tokenize(text, tokenizer, max_length=MAX_LENGTH):
    token_ids = tokenizer.encode(
        str(text),
        add_special_tokens=False
    )

    max_content_length = max_length - 2

    if len(token_ids) > max_content_length:
        first_part_length = max_content_length // 2
        last_part_length = max_content_length - first_part_length

        token_ids = (
            token_ids[:first_part_length] +
            token_ids[-last_part_length:]
        )

    input_ids = (
        [tokenizer.cls_token_id] +
        token_ids +
        [tokenizer.sep_token_id]
    )

    attention_mask = [1] * len(input_ids)

    padding_length = max_length - len(input_ids)

    if padding_length > 0:
        input_ids = input_ids + (
            [tokenizer.pad_token_id] * padding_length
        )
        attention_mask = attention_mask + ([0] * padding_length)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask
    }


class EmailInjectionDataset(Dataset):
    def __init__(self, dataframe, tokenizer, text_column, label_column):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.text_column = text_column
        self.label_column = label_column

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        encoded = head_tail_tokenize(
            row[self.text_column],
            self.tokenizer
        )

        return {
            "input_ids": torch.tensor(
                encoded["input_ids"],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                encoded["attention_mask"],
                dtype=torch.long
            ),
            "labels": torch.tensor(
                row[self.label_column],
                dtype=torch.long
            ),
            "sample_id": row["sample_id"],
            "context_id": row["context_id"]
        }


for sample_id in [
    "train_014_benign",
    "train_014_attack"
]:
    sample_text = train_df.loc[
        train_df["sample_id"] == sample_id,
        TEXT_COLUMN
    ].iloc[0]

    encoded_sample = head_tail_tokenize(
        sample_text,
        tokenizer
    )

    print(f"\nSample ID: {sample_id}")
    print("Final encoded length:", len(encoded_sample["input_ids"]))
    print(
        "Active tokens:",
        sum(encoded_sample["attention_mask"])
    )

In [ ]:
import random
import numpy as np
import torch

from torch.utils.data import DataLoader

PHASE_1_EPOCHS = 10
PHASE_2_EPOCHS = 5

HEAD_LEARNING_RATE = 5e-5
ENCODER_LEARNING_RATE = 1e-6
WEIGHT_DECAY = 0.01
ADAM_EPSILON = 1e-6
WARMUP_RATIO = 0.10


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def create_fold_loaders(dataframe, fold_number):
    fit_fold = dataframe[
        dataframe["cv_fold"] != fold_number
    ].copy()

    validation_fold = dataframe[
        dataframe["cv_fold"] == fold_number
    ].copy()

    fit_dataset = EmailInjectionDataset(
        fit_fold,
        tokenizer,
        TEXT_COLUMN,
        LABEL_COLUMN
    )

    validation_dataset = EmailInjectionDataset(
        validation_fold,
        tokenizer,
        TEXT_COLUMN,
        LABEL_COLUMN
    )

    generator = torch.Generator()
    generator.manual_seed(SEED + fold_number)

    fit_loader = DataLoader(
        fit_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=True
    )

    validation_loader = DataLoader(
        validation_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    return fit_fold, validation_fold, fit_loader, validation_loader


set_seed(SEED)

fit_fold_1, validation_fold_1, fit_loader_1, validation_loader_1 = (
    create_fold_loaders(train_df_cv, fold_number=1)
)

first_batch = next(iter(fit_loader_1))

print("Fold 1 fit shape:", fit_fold_1.shape)
print("Fold 1 validation shape:", validation_fold_1.shape)

print("\nFit label distribution:")
print(fit_fold_1["label_name"].value_counts())

print("\nValidation label distribution:")
print(validation_fold_1["label_name"].value_counts())

print("\nTraining batches:", len(fit_loader_1))
print("Validation batches:", len(validation_loader_1))

print("\nInput IDs shape:", first_batch["input_ids"].shape)
print("Attention mask shape:", first_batch["attention_mask"].shape)
print("Labels shape:", first_batch["labels"].shape)

print(
    "\nContext overlap:",
    len(
        set(fit_fold_1[GROUP_COLUMN]).intersection(
            set(validation_fold_1[GROUP_COLUMN])
        )
    )
)

In [ ]:
import json
import math
import time
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score
)
from transformers import (
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


def create_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    model.config.pad_token_id = tokenizer.pad_token_id

    return model.to(device)


def train_one_phase(model, data_loader, optimizer, scheduler, epochs):
    phase_history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

            if not torch.isfinite(loss):
                raise RuntimeError(
                    "Non-finite loss detected during training."
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

        average_loss = total_loss / len(data_loader)

        phase_history.append({
            "epoch": epoch,
            "train_loss": average_loss
        })

    return phase_history


def evaluate_model(model, data_loader):
    model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []
    all_sample_ids = []
    all_context_ids = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            probabilities = torch.softmax(
                outputs.logits,
                dim=1
            )[:, 1]

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
            all_sample_ids.extend(batch["sample_id"])
            all_context_ids.extend(batch["context_id"])

    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    all_probabilities = np.array(all_probabilities)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_predictions,
        labels=[0, 1],
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1]
    ).ravel()

    metrics = {
        "accuracy": accuracy_score(all_labels, all_predictions),
        "macro_f1": f1.mean(),
        "benign_f1": f1[0],
        "attack_f1": f1[1],
        "attack_precision": precision[1],
        "attack_recall": recall[1],
        "false_positive_rate": (
            fp / (fp + tn) if (fp + tn) > 0 else 0.0
        ),
        "roc_auc": roc_auc_score(
            all_labels,
            all_probabilities
        ),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    }

    predictions_dataframe = pd.DataFrame({
        "sample_id": all_sample_ids,
        "context_id": all_context_ids,
        "true_label": all_labels,
        "prediction": all_predictions,
        "attack_probability": all_probabilities
    })

    return metrics, predictions_dataframe


def create_optimizer_and_scheduler(
    model,
    head_only,
    training_steps
):
    no_decay_terms = ["bias", "LayerNorm.weight"]

    parameter_groups = []

    for name, parameter in model.named_parameters():
        is_encoder_parameter = name.startswith("deberta.")

        if head_only and is_encoder_parameter:
            parameter.requires_grad = False
            continue

        parameter.requires_grad = True

        learning_rate = (
            HEAD_LEARNING_RATE
            if not is_encoder_parameter
            else ENCODER_LEARNING_RATE
        )

        weight_decay = (
            0.0
            if any(term in name for term in no_decay_terms)
            else WEIGHT_DECAY
        )

        parameter_groups.append({
            "params": [parameter],
            "lr": learning_rate,
            "weight_decay": weight_decay
        })

    optimizer = torch.optim.AdamW(
        parameter_groups,
        eps=ADAM_EPSILON,
        foreach=False,
        fused=False
    )

    warmup_steps = int(WARMUP_RATIO * training_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=training_steps
    )

    return optimizer, scheduler


all_fold_metrics = []
all_fold_predictions = []
all_training_history = []

start_time = time.time()

for fold_number in range(1, N_SPLITS + 1):
    print(f"\n{'=' * 60}")
    print(f"Starting fold {fold_number}/{N_SPLITS}")
    print(f"{'=' * 60}")

    set_seed(SEED + fold_number)

    _, _, fit_loader, validation_loader = create_fold_loaders(
        train_df_cv,
        fold_number
    )

    fold_model = create_model()

    phase_1_steps = len(fit_loader) * PHASE_1_EPOCHS

    phase_1_optimizer, phase_1_scheduler = (
        create_optimizer_and_scheduler(
            fold_model,
            head_only=True,
            training_steps=phase_1_steps
        )
    )

    phase_1_history = train_one_phase(
        fold_model,
        fit_loader,
        phase_1_optimizer,
        phase_1_scheduler,
        PHASE_1_EPOCHS
    )

    phase_2_steps = len(fit_loader) * PHASE_2_EPOCHS

    phase_2_optimizer, phase_2_scheduler = (
        create_optimizer_and_scheduler(
            fold_model,
            head_only=False,
            training_steps=phase_2_steps
        )
    )

    phase_2_history = train_one_phase(
        fold_model,
        fit_loader,
        phase_2_optimizer,
        phase_2_scheduler,
        PHASE_2_EPOCHS
    )

    fold_metrics, fold_predictions = evaluate_model(
        fold_model,
        validation_loader
    )

    fold_metrics["fold"] = fold_number
    all_fold_metrics.append(fold_metrics)

    fold_predictions["fold"] = fold_number
    all_fold_predictions.append(fold_predictions)

    for row in phase_1_history:
        row["fold"] = fold_number
        row["phase"] = "head_only"
        all_training_history.append(row)

    for row in phase_2_history:
        row["fold"] = fold_number
        row["phase"] = "full_finetuning"
        all_training_history.append(row)

    print(
        f"Fold {fold_number} results | "
        f"Macro-F1: {fold_metrics['macro_f1']:.4f} | "
        f"Attack Recall: {fold_metrics['attack_recall']:.4f} | "
        f"FPR: {fold_metrics['false_positive_rate']:.4f} | "
        f"ROC-AUC: {fold_metrics['roc_auc']:.4f}"
    )

    del fold_model
    torch.cuda.empty_cache()

fold_metrics_table = pd.DataFrame(all_fold_metrics).sort_values(
    "fold"
).reset_index(drop=True)

oof_predictions = pd.concat(
    all_fold_predictions,
    ignore_index=True
).sort_values("sample_id").reset_index(drop=True)

training_history_table = pd.DataFrame(
    all_training_history
).sort_values(["fold", "phase", "epoch"]).reset_index(drop=True)

metric_columns = [
    "accuracy",
    "macro_f1",
    "benign_f1",
    "attack_f1",
    "attack_precision",
    "attack_recall",
    "false_positive_rate",
    "roc_auc"
]

summary_table = fold_metrics_table[metric_columns].agg(
    ["mean", "std"]
).T.reset_index()

summary_table.columns = [
    "metric",
    "mean",
    "standard_deviation"
]

FOLD_METRICS_PATH = (
    CV_RESULTS_DIR / "deberta_grouped_cv_fold_metrics.csv"
)

OOF_PREDICTIONS_PATH = (
    CV_RESULTS_DIR / "deberta_grouped_cv_oof_predictions.csv"
)

TRAINING_HISTORY_PATH = (
    CV_RESULTS_DIR / "deberta_grouped_cv_training_history.csv"
)

SUMMARY_PATH = (
    CV_RESULTS_DIR / "deberta_grouped_cv_summary.csv"
)

CONFIG_PATH = (
    CV_RESULTS_DIR / "deberta_grouped_cv_config.json"
)

fold_metrics_table.to_csv(FOLD_METRICS_PATH, index=False)
oof_predictions.to_csv(OOF_PREDICTIONS_PATH, index=False)
training_history_table.to_csv(TRAINING_HISTORY_PATH, index=False)
summary_table.to_csv(SUMMARY_PATH, index=False)

with open(CONFIG_PATH, "w") as file:
    json.dump(
        {
            "model_name": MODEL_NAME,
            "evaluation_protocol": "5-fold StratifiedGroupKFold",
            "group_column": GROUP_COLUMN,
            "seed": SEED,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "phase_1": {
                "encoder_frozen": True,
                "epochs": PHASE_1_EPOCHS,
                "head_learning_rate": HEAD_LEARNING_RATE
            },
            "phase_2": {
                "encoder_frozen": False,
                "epochs": PHASE_2_EPOCHS,
                "encoder_learning_rate": ENCODER_LEARNING_RATE,
                "head_learning_rate": HEAD_LEARNING_RATE
            },
            "warmup_ratio": WARMUP_RATIO,
            "weight_decay": WEIGHT_DECAY,
            "optimizer": "AdamW"
        },
        file,
        indent=2
    )

print(f"\nCompleted in {(time.time() - start_time) / 60:.1f} minutes.")

print("\nFold-level metrics:")
display(fold_metrics_table)

print("\nCross-validation summary:")
display(summary_table)

print(f"\nSaved fold metrics: {FOLD_METRICS_PATH}")
print(f"Saved OOF predictions: {OOF_PREDICTIONS_PATH}")
print(f"Saved summary: {SUMMARY_PATH}")

In [ ]:
attack_metadata = train_df_cv[
    train_df_cv[LABEL_COLUMN] == 1
][
    [
        "context_id",
        "attack_category",
        "attack_number",
        "cv_fold"
    ]
].copy()

print("Attack categories in the cross-validation corpus:")
display(
    attack_metadata["attack_category"]
    .value_counts()
    .rename_axis("attack_category")
    .reset_index(name="count")
)

print("\nAttack categories by validation fold:")
category_by_fold = pd.crosstab(
    attack_metadata["attack_category"],
    attack_metadata["cv_fold"]
)

display(category_by_fold)

template_overlap_rows = []

for fold_number in range(1, N_SPLITS + 1):
    fit_categories = set(
        attack_metadata.loc[
            attack_metadata["cv_fold"] != fold_number,
            "attack_category"
        ].dropna()
    )

    validation_categories = set(
        attack_metadata.loc[
            attack_metadata["cv_fold"] == fold_number,
            "attack_category"
        ].dropna()
    )

    shared_categories = fit_categories.intersection(
        validation_categories
    )

    unseen_validation_categories = validation_categories.difference(
        fit_categories
    )

    template_overlap_rows.append({
        "fold": fold_number,
        "validation_categories": len(validation_categories),
        "shared_with_fit": len(shared_categories),
        "unseen_in_fit": len(unseen_validation_categories),
        "shared_category_names": " | ".join(
            sorted(shared_categories)
        ),
        "unseen_category_names": " | ".join(
            sorted(unseen_validation_categories)
        )
    })

template_overlap_table = pd.DataFrame(template_overlap_rows)

TEMPLATE_OVERLAP_PATH = (
    CV_RESULTS_DIR / "grouped_cv_attack_category_overlap.csv"
)

template_overlap_table.to_csv(
    TEMPLATE_OVERLAP_PATH,
    index=False
)

print("\nAttack-category overlap between fit and validation data:")
display(template_overlap_table)

print(f"\nSaved overlap analysis: {TEMPLATE_OVERLAP_PATH}")

In [ ]:
from sklearn.model_selection import GroupKFold

CATEGORY_CV_RESULTS_DIR = (
    PROJECT_DIR / "results" / "category_disjoint_cv_deberta"
)

CATEGORY_CV_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

context_attack_category = (
    train_df[train_df[LABEL_COLUMN] == 1]
    .set_index("context_id")["attack_category"]
    .to_dict()
)

train_df_category_cv = train_df.copy()

train_df_category_cv["attack_category_group"] = (
    train_df_category_cv["context_id"].map(
        context_attack_category
    )
)

assert train_df_category_cv[
    "attack_category_group"
].notna().all()

category_splitter = GroupKFold(n_splits=N_SPLITS)

category_fold_assignments = pd.Series(
    index=train_df_category_cv.index,
    dtype="int64"
)

category_fold_rows = []

for fold_number, (fit_indices, validation_indices) in enumerate(
    category_splitter.split(
        X=train_df_category_cv[TEXT_COLUMN],
        y=train_df_category_cv[LABEL_COLUMN],
        groups=train_df_category_cv["attack_category_group"]
    ),
    start=1
):
    category_fold_assignments.loc[validation_indices] = fold_number

    fit_fold = train_df_category_cv.iloc[fit_indices]
    validation_fold = train_df_category_cv.iloc[
        validation_indices
    ]

    fit_categories = set(
        fit_fold["attack_category_group"]
    )

    validation_categories = set(
        validation_fold["attack_category_group"]
    )

    shared_categories = fit_categories.intersection(
        validation_categories
    )

    category_fold_rows.append({
        "fold": fold_number,
        "fit_rows": len(fit_fold),
        "validation_rows": len(validation_fold),
        "fit_contexts": fit_fold["context_id"].nunique(),
        "validation_contexts": validation_fold[
            "context_id"
        ].nunique(),
        "validation_benign": int(
            (validation_fold[LABEL_COLUMN] == 0).sum()
        ),
        "validation_attack": int(
            (validation_fold[LABEL_COLUMN] == 1).sum()
        ),
        "fit_attack_categories": len(fit_categories),
        "validation_attack_categories": len(
            validation_categories
        ),
        "shared_attack_categories": len(shared_categories),
        "validation_category_names": " | ".join(
            sorted(validation_categories)
        )
    })

train_df_category_cv["cv_fold"] = category_fold_assignments

category_fold_summary = pd.DataFrame(category_fold_rows)

CATEGORY_FOLD_ASSIGNMENTS_PATH = (
    CATEGORY_CV_RESULTS_DIR /
    "category_disjoint_fold_assignments.csv"
)

CATEGORY_FOLD_SUMMARY_PATH = (
    CATEGORY_CV_RESULTS_DIR /
    "category_disjoint_fold_summary.csv"
)

train_df_category_cv.to_csv(
    CATEGORY_FOLD_ASSIGNMENTS_PATH,
    index=False
)

category_fold_summary.to_csv(
    CATEGORY_FOLD_SUMMARY_PATH,
    index=False
)

print("Category-disjoint folds created successfully.")
display(category_fold_summary)

print("\nSamples assigned to each validation fold:")
print(
    train_df_category_cv["cv_fold"]
    .value_counts()
    .sort_index()
)

print(f"\nSaved fold assignments: {CATEGORY_FOLD_ASSIGNMENTS_PATH}")
print(f"Saved fold summary: {CATEGORY_FOLD_SUMMARY_PATH}")

In [ ]:
train_df_cv = train_df_category_cv.copy()
CV_RESULTS_DIR = CATEGORY_CV_RESULTS_DIR

assert (
    category_fold_summary[
        "shared_attack_categories"
    ] == 0
).all()

print("Category-disjoint evaluation is active.")
print("Results directory:", CV_RESULTS_DIR)
print("Number of folds:", N_SPLITS)

In [ ]:
import json
import math
import time
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score
)
from transformers import (
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


def create_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    model.config.pad_token_id = tokenizer.pad_token_id

    return model.to(device)


def train_one_phase(model, data_loader, optimizer, scheduler, epochs):
    phase_history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

            if not torch.isfinite(loss):
                raise RuntimeError(
                    "Non-finite loss detected during training."
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

        average_loss = total_loss / len(data_loader)

        phase_history.append({
            "epoch": epoch,
            "train_loss": average_loss
        })

    return phase_history


def evaluate_model(model, data_loader):
    model.eval()

    all_labels = []
    all_predictions = []
    all_probabilities = []
    all_sample_ids = []
    all_context_ids = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            probabilities = torch.softmax(
                outputs.logits,
                dim=1
            )[:, 1]

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
            all_sample_ids.extend(batch["sample_id"])
            all_context_ids.extend(batch["context_id"])

    all_labels = np.array(all_labels)
    all_predictions = np.array(all_predictions)
    all_probabilities = np.array(all_probabilities)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_predictions,
        labels=[0, 1],
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        all_labels,
        all_predictions,
        labels=[0, 1]
    ).ravel()

    metrics = {
        "accuracy": accuracy_score(all_labels, all_predictions),
        "macro_f1": f1.mean(),
        "benign_f1": f1[0],
        "attack_f1": f1[1],
        "attack_precision": precision[1],
        "attack_recall": recall[1],
        "false_positive_rate": (
            fp / (fp + tn) if (fp + tn) > 0 else 0.0
        ),
        "roc_auc": roc_auc_score(
            all_labels,
            all_probabilities
        ),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    }

    predictions_dataframe = pd.DataFrame({
        "sample_id": all_sample_ids,
        "context_id": all_context_ids,
        "true_label": all_labels,
        "prediction": all_predictions,
        "attack_probability": all_probabilities
    })

    return metrics, predictions_dataframe


def create_optimizer_and_scheduler(
    model,
    head_only,
    training_steps
):
    no_decay_terms = ["bias", "LayerNorm.weight"]

    parameter_groups = []

    for name, parameter in model.named_parameters():
        is_encoder_parameter = name.startswith("deberta.")

        if head_only and is_encoder_parameter:
            parameter.requires_grad = False
            continue

        parameter.requires_grad = True

        learning_rate = (
            HEAD_LEARNING_RATE
            if not is_encoder_parameter
            else ENCODER_LEARNING_RATE
        )

        weight_decay = (
            0.0
            if any(term in name for term in no_decay_terms)
            else WEIGHT_DECAY
        )

        parameter_groups.append({
            "params": [parameter],
            "lr": learning_rate,
            "weight_decay": weight_decay
        })

    optimizer = torch.optim.AdamW(
        parameter_groups,
        eps=ADAM_EPSILON,
        foreach=False,
        fused=False
    )

    warmup_steps = int(WARMUP_RATIO * training_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=training_steps
    )

    return optimizer, scheduler


all_fold_metrics = []
all_fold_predictions = []
all_training_history = []

start_time = time.time()

for fold_number in range(1, N_SPLITS + 1):
    print(f"\n{'=' * 60}")
    print(f"Starting fold {fold_number}/{N_SPLITS}")
    print(f"{'=' * 60}")

    set_seed(SEED + fold_number)

    _, _, fit_loader, validation_loader = create_fold_loaders(
        train_df_cv,
        fold_number
    )

    fold_model = create_model()

    phase_1_steps = len(fit_loader) * PHASE_1_EPOCHS

    phase_1_optimizer, phase_1_scheduler = (
        create_optimizer_and_scheduler(
            fold_model,
            head_only=True,
            training_steps=phase_1_steps
        )
    )

    phase_1_history = train_one_phase(
        fold_model,
        fit_loader,
        phase_1_optimizer,
        phase_1_scheduler,
        PHASE_1_EPOCHS
    )

    phase_2_steps = len(fit_loader) * PHASE_2_EPOCHS

    phase_2_optimizer, phase_2_scheduler = (
        create_optimizer_and_scheduler(
            fold_model,
            head_only=False,
            training_steps=phase_2_steps
        )
    )

    phase_2_history = train_one_phase(
        fold_model,
        fit_loader,
        phase_2_optimizer,
        phase_2_scheduler,
        PHASE_2_EPOCHS
    )

    fold_metrics, fold_predictions = evaluate_model(
        fold_model,
        validation_loader
    )

    fold_metrics["fold"] = fold_number
    all_fold_metrics.append(fold_metrics)

    fold_predictions["fold"] = fold_number
    all_fold_predictions.append(fold_predictions)

    for row in phase_1_history:
        row["fold"] = fold_number
        row["phase"] = "head_only"
        all_training_history.append(row)

    for row in phase_2_history:
        row["fold"] = fold_number
        row["phase"] = "full_finetuning"
        all_training_history.append(row)

    print(
        f"Fold {fold_number} results | "
        f"Macro-F1: {fold_metrics['macro_f1']:.4f} | "
        f"Attack Recall: {fold_metrics['attack_recall']:.4f} | "
        f"FPR: {fold_metrics['false_positive_rate']:.4f} | "
        f"ROC-AUC: {fold_metrics['roc_auc']:.4f}"
    )

    del fold_model
    torch.cuda.empty_cache()

fold_metrics_table = pd.DataFrame(all_fold_metrics).sort_values(
    "fold"
).reset_index(drop=True)

oof_predictions = pd.concat(
    all_fold_predictions,
    ignore_index=True
).sort_values("sample_id").reset_index(drop=True)

training_history_table = pd.DataFrame(
    all_training_history
).sort_values(["fold", "phase", "epoch"]).reset_index(drop=True)

metric_columns = [
    "accuracy",
    "macro_f1",
    "benign_f1",
    "attack_f1",
    "attack_precision",
    "attack_recall",
    "false_positive_rate",
    "roc_auc"
]

summary_table = fold_metrics_table[metric_columns].agg(
    ["mean", "std"]
).T.reset_index()

summary_table.columns = [
    "metric",
    "mean",
    "standard_deviation"
]

FOLD_METRICS_PATH = (
    CV_RESULTS_DIR / "deberta_grouped_cv_fold_metrics.csv"
)

OOF_PREDICTIONS_PATH = (
    CV_RESULTS_DIR / "deberta_grouped_cv_oof_predictions.csv"
)

TRAINING_HISTORY_PATH = (
    CV_RESULTS_DIR / "deberta_grouped_cv_training_history.csv"
)

SUMMARY_PATH = (
    CV_RESULTS_DIR / "deberta_grouped_cv_summary.csv"
)

CONFIG_PATH = (
    CV_RESULTS_DIR / "deberta_grouped_cv_config.json"
)

fold_metrics_table.to_csv(FOLD_METRICS_PATH, index=False)
oof_predictions.to_csv(OOF_PREDICTIONS_PATH, index=False)
training_history_table.to_csv(TRAINING_HISTORY_PATH, index=False)
summary_table.to_csv(SUMMARY_PATH, index=False)

with open(CONFIG_PATH, "w") as file:
    json.dump(
        {
            "model_name": MODEL_NAME,
            "evaluation_protocol": "5-fold StratifiedGroupKFold",
            "group_column": GROUP_COLUMN,
            "seed": SEED,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "phase_1": {
                "encoder_frozen": True,
                "epochs": PHASE_1_EPOCHS,
                "head_learning_rate": HEAD_LEARNING_RATE
            },
            "phase_2": {
                "encoder_frozen": False,
                "epochs": PHASE_2_EPOCHS,
                "encoder_learning_rate": ENCODER_LEARNING_RATE,
                "head_learning_rate": HEAD_LEARNING_RATE
            },
            "warmup_ratio": WARMUP_RATIO,
            "weight_decay": WEIGHT_DECAY,
            "optimizer": "AdamW"
        },
        file,
        indent=2
    )

print(f"\nCompleted in {(time.time() - start_time) / 60:.1f} minutes.")

print("\nFold-level metrics:")
display(fold_metrics_table)

print("\nCross-validation summary:")
display(summary_table)

print(f"\nSaved fold metrics: {FOLD_METRICS_PATH}")
print(f"Saved OOF predictions: {OOF_PREDICTIONS_PATH}")
print(f"Saved summary: {SUMMARY_PATH}")

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupKFold

CATEGORY_CV_RESULTS_DIR = (
    PROJECT_DIR / "results" / "category_disjoint_cv_deberta"
)
CATEGORY_CV_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

N_SPLITS = 5
train_df = pd.read_csv(TRAIN_PATH)

context_attack_category = (
    train_df[train_df[LABEL_COLUMN] == 1]
    .set_index(GROUP_COLUMN)["attack_category"]
    .to_dict()
)

train_df_category_cv = train_df.copy()
train_df_category_cv["attack_category_group"] = (
    train_df_category_cv[GROUP_COLUMN].map(context_attack_category)
)

category_splitter = GroupKFold(n_splits=N_SPLITS)
category_fold_assignments = pd.Series(
    index=train_df_category_cv.index,
    dtype="int64",
)

for fold_number, (_, validation_indices) in enumerate(
    category_splitter.split(
        X=train_df_category_cv[TEXT_COLUMN],
        y=train_df_category_cv[LABEL_COLUMN],
        groups=train_df_category_cv["attack_category_group"],
    ),
    start=1,
):
    category_fold_assignments.loc[validation_indices] = fold_number

train_df_category_cv["cv_fold"] = category_fold_assignments

print("Training shape:", train_df_category_cv.shape)
print("Results directory:", CATEGORY_CV_RESULTS_DIR)
print("Samples in each fold:")
print(train_df_category_cv["cv_fold"].value_counts().sort_index())

for fold_number in range(1, N_SPLITS + 1):
    fit_categories = set(
        train_df_category_cv.loc[
            train_df_category_cv["cv_fold"] != fold_number,
            "attack_category_group",
        ]
    )
    validation_categories = set(
        train_df_category_cv.loc[
            train_df_category_cv["cv_fold"] == fold_number,
            "attack_category_group",
        ]
    )
    assert not fit_categories.intersection(validation_categories)

print("Category-disjoint evaluation is active.")


In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    roc_auc_score
)

SVM_RESULTS_DIR = (
    CATEGORY_CV_RESULTS_DIR / "linear_svm_unigram"
)

SVM_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

svm_fold_metrics = []
svm_oof_predictions = []

for fold_number in range(1, N_SPLITS + 1):
    fit_fold = train_df_category_cv[
        train_df_category_cv["cv_fold"] != fold_number
    ].copy()

    validation_fold = train_df_category_cv[
        train_df_category_cv["cv_fold"] == fold_number
    ].copy()

    X_fit = fit_fold[TEXT_COLUMN].fillna("").astype(str)
    y_fit = fit_fold[LABEL_COLUMN]

    X_validation = (
        validation_fold[TEXT_COLUMN]
        .fillna("")
        .astype(str)
    )

    y_validation = validation_fold[LABEL_COLUMN]

    linear_svm_pipeline = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                analyzer="word",
                ngram_range=(1, 1),
                norm="l2"
            )
        ),
        (
            "classifier",
            LinearSVC(
                class_weight="balanced",
                random_state=SEED
            )
        )
    ])

    linear_svm_pipeline.fit(X_fit, y_fit)

    predictions = linear_svm_pipeline.predict(X_validation)

    decision_scores = linear_svm_pipeline.decision_function(
        X_validation
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_validation,
        predictions,
        labels=[0, 1],
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_validation,
        predictions,
        labels=[0, 1]
    ).ravel()

    fold_metrics = {
        "fold": fold_number,
        "accuracy": accuracy_score(
            y_validation,
            predictions
        ),
        "macro_f1": f1.mean(),
        "benign_f1": f1[0],
        "attack_f1": f1[1],
        "attack_precision": precision[1],
        "attack_recall": recall[1],
        "false_positive_rate": (
            fp / (fp + tn) if (fp + tn) > 0 else 0.0
        ),
        "roc_auc": roc_auc_score(
            y_validation,
            decision_scores
        ),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp)
    }

    svm_fold_metrics.append(fold_metrics)

    fold_predictions = validation_fold[
        [
            "sample_id",
            "context_id",
            "attack_category",
            LABEL_COLUMN
        ]
    ].copy()

    fold_predictions["fold"] = fold_number
    fold_predictions["prediction"] = predictions
    fold_predictions["decision_score"] = decision_scores

    svm_oof_predictions.append(fold_predictions)

    print(
        f"Fold {fold_number} results | "
        f"Macro-F1: {fold_metrics['macro_f1']:.4f} | "
        f"Attack Recall: {fold_metrics['attack_recall']:.4f} | "
        f"FPR: {fold_metrics['false_positive_rate']:.4f} | "
        f"ROC-AUC: {fold_metrics['roc_auc']:.4f}"
    )

svm_fold_metrics_table = pd.DataFrame(
    svm_fold_metrics
).sort_values("fold").reset_index(drop=True)

svm_oof_predictions_table = pd.concat(
    svm_oof_predictions,
    ignore_index=True
).sort_values("sample_id").reset_index(drop=True)

svm_metric_columns = [
    "accuracy",
    "macro_f1",
    "benign_f1",
    "attack_f1",
    "attack_precision",
    "attack_recall",
    "false_positive_rate",
    "roc_auc"
]

svm_summary_table = svm_fold_metrics_table[
    svm_metric_columns
].agg(["mean", "std"]).T.reset_index()

svm_summary_table.columns = [
    "metric",
    "mean",
    "standard_deviation"
]

SVM_FOLD_METRICS_PATH = (
    SVM_RESULTS_DIR / "linear_svm_unigram_fold_metrics.csv"
)

SVM_OOF_PREDICTIONS_PATH = (
    SVM_RESULTS_DIR / "linear_svm_unigram_oof_predictions.csv"
)

SVM_SUMMARY_PATH = (
    SVM_RESULTS_DIR / "linear_svm_unigram_summary.csv"
)

SVM_CONFIG_PATH = (
    SVM_RESULTS_DIR / "linear_svm_unigram_config.json"
)

svm_fold_metrics_table.to_csv(
    SVM_FOLD_METRICS_PATH,
    index=False
)

svm_oof_predictions_table.to_csv(
    SVM_OOF_PREDICTIONS_PATH,
    index=False
)

svm_summary_table.to_csv(
    SVM_SUMMARY_PATH,
    index=False
)

with open(SVM_CONFIG_PATH, "w") as file:
    json.dump(
        {
            "model": "LinearSVC",
            "class_weight": "balanced",
            "random_state": SEED,
            "tfidf": {
                "lowercase": True,
                "analyzer": "word",
                "ngram_range": [1, 1],
                "norm": "l2"
            },
            "evaluation_protocol": (
                "5-fold category-disjoint "
                "grouped cross-validation"
            )
        },
        file,
        indent=2
    )

print("\nLinear SVM fold-level metrics:")
display(svm_fold_metrics_table)

print("\nLinear SVM cross-validation summary:")
display(svm_summary_table)

print(f"\nSaved fold metrics: {SVM_FOLD_METRICS_PATH}")
print(f"Saved OOF predictions: {SVM_OOF_PREDICTIONS_PATH}")
print(f"Saved summary: {SVM_SUMMARY_PATH}")

In [ ]:
import pandas as pd

DEBERTA_SUMMARY_PATH = (
    CATEGORY_CV_RESULTS_DIR /
    "deberta_grouped_cv_summary.csv"
)

SVM_SUMMARY_PATH = (
    CATEGORY_CV_RESULTS_DIR /
    "linear_svm_unigram" /
    "linear_svm_unigram_summary.csv"
)

deberta_summary = pd.read_csv(DEBERTA_SUMMARY_PATH)

svm_summary = pd.read_csv(SVM_SUMMARY_PATH)

comparison_table = deberta_summary.merge(
    svm_summary,
    on="metric",
    suffixes=("_deberta", "_linear_svm")
)

comparison_table["mean_difference_deberta_minus_svm"] = (
    comparison_table["mean_deberta"] -
    comparison_table["mean_linear_svm"]
)

key_metrics = [
    "accuracy",
    "macro_f1",
    "attack_recall",
    "false_positive_rate",
    "roc_auc"
]

comparison_table = comparison_table[
    comparison_table["metric"].isin(key_metrics)
].copy()

comparison_table = comparison_table[
    [
        "metric",
        "mean_deberta",
        "standard_deviation_deberta",
        "mean_linear_svm",
        "standard_deviation_linear_svm",
        "mean_difference_deberta_minus_svm"
    ]
]

comparison_table = comparison_table.sort_values(
    "metric"
).reset_index(drop=True)

COMPARISON_PATH = (
    CATEGORY_CV_RESULTS_DIR /
    "deberta_vs_linear_svm_comparison.csv"
)

comparison_table.to_csv(
    COMPARISON_PATH,
    index=False
)

print("Category-disjoint cross-validation comparison:")
display(comparison_table.round(4))

print(f"\nSaved comparison table: {COMPARISON_PATH}")

In [ ]:
DEBERTA_OOF_PATH = (
    CATEGORY_CV_RESULTS_DIR /
    "deberta_grouped_cv_oof_predictions.csv"
)

SVM_OOF_PATH = (
    CATEGORY_CV_RESULTS_DIR /
    "linear_svm_unigram" /
    "linear_svm_unigram_oof_predictions.csv"
)

deberta_oof = pd.read_csv(DEBERTA_OOF_PATH)
svm_oof = pd.read_csv(SVM_OOF_PATH)

metadata_columns = [
    "sample_id",
    "context_id",
    "label_name",
    "attack_category_group"
]

sample_metadata = train_df_category_cv[
    metadata_columns
].copy()

fold_two_analysis = deberta_oof.merge(
    sample_metadata,
    on=["sample_id", "context_id"],
    how="left"
).merge(
    svm_oof[
        [
            "sample_id",
            "prediction",
            "decision_score"
        ]
    ].rename(
        columns={
            "prediction": "svm_prediction",
            "decision_score": "svm_decision_score"
        }
    ),
    on="sample_id",
    how="left"
)

fold_two_analysis = fold_two_analysis[
    fold_two_analysis["fold"] == 2
].copy()

fold_two_analysis["deberta_prediction_name"] = np.where(
    fold_two_analysis["prediction"] == 1,
    "attack",
    "benign"
)

fold_two_analysis["svm_prediction_name"] = np.where(
    fold_two_analysis["svm_prediction"] == 1,
    "attack",
    "benign"
)

fold_two_summary = fold_two_analysis.groupby(
    ["label_name", "attack_category_group"],
    dropna=False
).agg(
    samples=("sample_id", "count"),
    deberta_attacks_predicted=("prediction", "sum"),
    svm_attacks_predicted=("svm_prediction", "sum"),
    mean_deberta_attack_probability=(
        "attack_probability",
        "mean"
    ),
    mean_svm_decision_score=(
        "svm_decision_score",
        "mean"
    )
).reset_index()

FOLD_TWO_ANALYSIS_PATH = (
    CATEGORY_CV_RESULTS_DIR /
    "fold_2_over_defense_analysis.csv"
)

fold_two_analysis.to_csv(
    FOLD_TWO_ANALYSIS_PATH,
    index=False
)

print("Fold 2 category-level analysis:")
display(fold_two_summary.round(4))

print("\nFold 2 sample-level predictions:")
display(
    fold_two_analysis[
        [
            "sample_id",
            "label_name",
            "attack_category_group",
            "attack_probability",
            "deberta_prediction_name",
            "svm_decision_score",
            "svm_prediction_name"
        ]
    ].sort_values(
        ["label_name", "sample_id"]
    ).reset_index(drop=True)
)

print(f"\nSaved analysis: {FOLD_TWO_ANALYSIS_PATH}")